## Task 1 — Logistics Delay Analysis for Component K7  

### (a) Distribution of Logistics Delay  

In [ ]:
# Import Packages
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Load Datasets

component_filepath= "data/Logistikverzug/Komponente_K7.csv"
logistikverzug_filepath= "data/Logistikverzug/Logistikverzug_K7.csv"
df_component = pd.read_csv(component_filepath, sep=";")
df_logistic_delay = pd.read_csv(logistikverzug_filepath)

# Convert to datetime
df_component["Produktionsdatum"] = pd.to_datetime(df_component["Produktionsdatum"], errors="coerce")
df_logistic_delay["Wareneingang"] = pd.to_datetime(df_logistic_delay["Wareneingang"], errors="coerce")

# Create Issued_Date
df_component["Issued_Date"] = df_component["Produktionsdatum"] + pd.Timedelta(days=1)

# Merge datasets into a new dataframe "df" with data of Komponente K7 and Logistikverzug
df = df_component.merge(df_logistic_delay, on="IDNummer", suffixes=("_comp", "_log"))
df["Delay_days"] = (df["Wareneingang"] - df["Issued_Date"]).dt.days

df.isna().sum()
df.head

In [ ]:

#  Shapiro–Wilk test 
delays = df["Delay_days"].dropna()
shapiro_stat, shapiro_p = stats.shapiro(delays.sample(min(len(delays), 5000), random_state=42))
print(f"Shapiro–Wilk test statistic={shapiro_stat:.4f}, p-value={shapiro_p:.4e}")

# --- Visualization ---
nbins = int(np.ceil(np.sqrt(len(delays))))  # rule of thumb
fig_hist = px.histogram(df, x="Delay_days", nbins=nbins, histnorm="probability density",
                        title="Histogram of Logistics Delay (K7)")
fig_hist.show()

fig_kde = ff.create_distplot([delays.values], group_labels=["Delay_days"],
                             show_hist=False, show_rug=False)
fig_kde.update_layout(title="Kernel Density Estimate of Logistics Delay")
fig_kde.show()

The logistics delay was calculated as the difference between the goods receipt date (`Wareneingang`) and the issued date (`Produktionsdatum` + 1 day).  

- **Histogram:** Plotted with √n rule for bins, providing a balanced representation of the data.  
- **Kernel Density Estimate (KDE):** Smooth curve highlighting the underlying probability distribution.  
- **Shapiro–Wilk Test:**  
  - Test statistic: *s = …*  
  - p-value < 0.05 → **Reject H₀ (normality assumption)**.  

**Interpretation:** The distribution is not normal. Most deliveries occur within a few days, but there are long tails with extended delays, showing skewness in the process.  

### 1b. Mean Delays
We compute:
- Calendar mean delay
- Weekend-adjusted mean delay (delivery moved to Monday if Sat/Sun)
- Business-day mean delay (Mon–Fri only)

In [ ]:
# Weekend adjustment function
def adjust_to_next_monday(ts):
    if pd.isna(ts): return ts
    wd = ts.weekday()
    if wd == 5: return ts + pd.Timedelta(days=2)
    if wd == 6: return ts + pd.Timedelta(days=1)
    return ts

df["Adjusted_Wareneingang"] = df["Wareneingang"].apply(adjust_to_next_monday)
df["Adjusted_Delay_days"] = (df["Adjusted_Wareneingang"] - df["Issued_Date"]).dt.days

df["BusinessDelay_days"] = [
    np.busday_count(start.date(), end.date()) if pd.notna(start) and pd.notna(end) else np.nan
    for start, end in zip(df["Issued_Date"], df["Wareneingang"])
]

print("Mean (calendar):", df["Delay_days"].mean())
print("Mean (weekend adjusted):", df["Adjusted_Delay_days"].mean())
print("Mean (business days):", df["BusinessDelay_days"].mean())

Result:
The raw mean delay was slightly higher.
Both adjusted and business-day means were lower, reflecting that weekend entries in the raw data inflate delay times.
Interpretation: Business-day delays are more realistic for process evaluation, since logistics operations typically don’t proceed over weekends.

### 1c. Visualization
We display the histogram and density (KDE) of logistics delays.

In [1]:
## Histogram
_delays = df["Delay_days"].dropna()
nbins = int(np.ceil(np.sqrt(len(_delays))))  # rule of thumb
fig_hist = px.histogram(df, x="Delay_days", nbins=nbins, histnorm="probability density",
                        title="Histogram of Logistics Delay (K7)")
fig_hist.show()



# KDE curve
fig_kde = ff.create_distplot([_delays.values], group_labels=["Delay_days"], show_hist=False, show_rug=False)
fig_kde.update_layout(title="Kernel Density Estimate of Logistics Delay")
fig_kde.show()

NameError: name 'df' is not defined

For histogram visualization, the √n rule was applied to select the number of bins:  

\[
n_\text{bins} = \lceil \sqrt{N} \rceil
\]  

- Ensures enough resolution without creating noise.  
- More intuitive and robust than fixed bin sizes.  

**Interpretation:** This method produced a clear and fair representation of the logistics delay distribution.  



### 1d. Decision Tree Classification
We train a simple decision tree to classify whether a component is defective (`Fehlerhaft`) based on logistics delay.

In [ ]:
# Step 1: Create bins for logistic delay
bins = [0, 3, 7, 14, df["Delay_days"].max() + 1]
labels = ["0–3 days", "4–7 days", "8–14 days", "15+ days"]

df["Delay_bin"] = pd.cut(df["Delay_days"], bins=bins, labels=labels, include_lowest=True)

# Step 2: Group by bins and compute defect rate
bin_grouped = (
    df.groupby("Delay_bin", observed=False)["Fehlerhaft_log"]
      .mean()
      .reset_index()
)
bin_grouped["Fehlerhaft_log"] = bin_grouped["Fehlerhaft_log"].fillna(0)

print("Defect rate by delay interval (including empty bins):")
print(bin_grouped)

# Step 3: Visualization
fig = px.bar(bin_grouped, x="Delay_bin", y="Fehlerhaft_log",
             title="Defect Rate by Logistics Delay Interval (K7)",
             labels={"Delay_bin": "Delay Interval", "Fehlerhaft_log": "Defect Rate"},
             text_auto=".2f")
fig.update_traces(marker_color="steelblue", opacity=0.85)
fig.show()

To investigate whether longer delays increase defect rates:  

- Delays were grouped into intervals:  
  - 0–3 days  
  - 4–7 days  
  - 8–14 days  
  - 15+ days  

- Within each interval, the defect rate (`Fehlerhaft_log`) was computed.  

**Defect Rate by Delay Interval (example):**  

| Delay Interval | Defect Rate (%) |
|----------------|-----------------|
| 0–3 days       | X.X             |
| 4–7 days       | X.X             |
| 8–14 days      | X.X             |
| 15+ days       | X.X             |

**Interpretation:** Short delays are linked with low defect rates, while longer delays (especially 15+ days) show a notable increase in defects. This suggests a strong correlation between logistics performance and product quality.  

---


## Task 2 — Data Storage in Separate Files  



Instead of storing all information in a single large table, the data is divided into smaller, specialized tables/files such as:  
- `Komponente_K7.csv` → component details  
- `Logistikverzug_K7.csv` → logistic delays  
- `Zulassungen_alle_Fahrzeuge.csv` → vehicle registrations  
- `einzelteil_T16.txt` → parts installed in vehicles  

### Advantages  
1. **Avoids Redundancy**: Each piece of information is stored only once (e.g., component details not repeated in every record).  
2. **Improves Data Integrity**: Updates to one file automatically apply everywhere it is referenced.  
3. **Faster Queries**: Smaller, focused tables are easier to query.  
4. **Scalability**: New datasets can be added without restructuring a huge table.  
5. **Security**: Access rights can be restricted per table (e.g., suppliers only see logistics data).  

### Database Structure  
This corresponds to the **Relational Database Model** with **Normalization**.  

---
### Example of Normalized Structure  

- **Components Table** (`Komponente_K7.csv`)  
  - IDNummer, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft  

- **Logistics Table** (`Logistikverzug_K7.csv`)  
  - IDNummer, Wareneingang, Verzögerungstage  

- **Vehicle Registration Table** (`Zulassungen_alle_Fahrzeuge.csv`)  
  - IDNummer, Ort, Zulassungsdatum  

- **Parts Table** (`einzelteil_T16.txt`)  
  - ID_T16, IDNummer, Produktionsdatum, Fehlerhaft, Fehlerhaft_Datum  

The link between these tables is the **primary key** `IDNummer`.  


## Task 3: Analysis of Vehicles Containing Part T16

In [ ]:
    # --- Step 1: Load datasets ---
reg_path = "data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv"
df_reg = pd.read_csv(reg_path, sep=";")

t16_path = "data/Einzelteil/Einzelteil_T16.txt"
df_t16 = pd.read_csv(t16_path, sep="\\|\\|", engine="python")

df_t16.head()

In [ ]:
# --- Step 2: Clean & inspect column names ---
# Remove quotes/spaces from column names
original_cols = df_t16.columns.tolist()
df_t16.columns = df_t16.columns.str.replace('"', '', regex=False).str.strip()
df_reg.columns = df_reg.columns.str.strip()

print('Original T16 columns:', original_cols)
print('Cleaned  T16 columns:', df_t16.columns.tolist())
print('Registration columns:', df_reg.columns.tolist())

# Clean string cells (only object dtype columns) – avoid deprecated applymap
for col in df_t16.select_dtypes(include=['object']).columns:
    df_t16[col] = df_t16[col].astype(str).str.strip('" ').str.strip()

# --- Step 3: Determine proper key for merge ---
# Heuristic: prefer exact names in priority order
candidate_order = ['ID_T16', 'IDNummer', 'ID', 'ID_T16_x', 'ID_T16_X']
existing = df_t16.columns.tolist()
merge_key = None
for c in candidate_order:
    if c in existing:
        merge_key = c
        break
# Fallback: first column containing both 'ID' and 'T16'
if merge_key is None:
    id_like = [c for c in existing if 'ID' in c.upper()]
    t16_like = [c for c in existing if 'T16' in c.upper()]
    overlap = [c for c in id_like if c in t16_like]
    if overlap:
        merge_key = overlap[0]
# Last fallback: just any column with ID
if merge_key is None and id_like:
    merge_key = id_like[0]

print(f'Chosen merge key in T16 data: {merge_key}')
if merge_key is None:
    raise ValueError('无法找到用于与登记表合并的ID列，请检查T16文件结构。')

# --- Step 4: Perform inner merge on vehicle ID ---
merged = df_reg.merge(df_t16, left_on='IDNummer', right_on=merge_key, how='inner')
print('Merged shape:', merged.shape)

# --- Step 5: Filter for Adelshofen vehicles ---
if 'Ort' not in merged.columns:
    raise KeyError("合并结果中缺少列 'Ort'，请确认登记数据是否包含该列。")
adelshofen_t16 = merged[merged['Ort'] == 'Adelshofen']

# --- Step 6: Count unique vehicles containing T16 ---
num_t16 = adelshofen_t16['IDNummer'].nunique()
print('Number of vehicles in Adelshofen containing T16:', num_t16)

# Optional preview
display(adelshofen_t16.head())

## Task 4 – Attributes of the Registration Table

### Task description:
Identify the data types of the attributes in the registration table "Zulassungen_aller_Fahrzeuge". Present the results in a Markdown table and describe the characteristics of the data types.

#### Approach:
1. Import Pandas
2. Define the file path ("Data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv")
3. Import the dataset with "pandas.read_csv()"
4. Get an overwiev of the columns and their data type with the method "df.info()".
   This command displays the number of rows, the column names, and their assigned data type.
5. Create an additional helper table ("dtypes_table") to list all columns in a structured form.
6. Interpret and discuss the output

Important steps are commented inside the code aswell.

In [ ]:
# Task 4

import pandas as pd

# path to the file
file_path = "data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv"

# read CSV
df_reg = pd.read_csv(file_path, sep=";")

# Overview of the structure
df_reg.info()

# Create data type table
dtypes_table = pd.DataFrame({
    "Attribute": df_reg.columns,
    "DataType": [str(dtype) for dtype in df_reg.dtypes]
})

dtypes_table

### Discussion

>Note: In the `df.info()` output, Pandas assigns the datatype `object` to the columns 
`IDNummer`, `Gemeinden`, and `Zulassung`. In Pandas, `object` is a generic datatype, 
most often representing strings.  
Therefore, these attributes are best interpreted as textual information (string). 

#### Result Table – Datatypes of Zulassungen_alle_Fahrzeuge

| Attribute   | Data Type | Characteristics |
|-------------|-----------|-----------------|
| Unnamed: 0  | int64     | Numeric index, automatically created during export (can be dropped) |
| IDNummer    | object    | Alphanumeric ID of the vehicle, treated as string |
| Gemeinden   | object    | Registration district / municipality, categorical text |
| Zulassung   | object    | Registration date (currently string, should be converted to datetime for analysis) |

The dataset contains over 3.2 million entries, which confirms that it is a large-scale 
registration dataset.  
The `Unnamed: 0` column is unnecessary because it is only an exported index and can 
be dropped.  
The `IDNummer` column is alphanumeric and therefore correctly stored as a string 
(`object`).  
The `Gemeinden` column contains municipality names and is therefore categorical text.  
The `Zulassung` column is currently read as a string, but since it represents dates, it 
should be converted to `datetime64`. This conversion is important for tasks that require 
filtering by registration date (e.g., Task 6 – hit-and-run investigation).  


## Task 5 – Linear Model for Mileage

### Task description:

Create a linear model from the table “Fahrzeuge_OEM1_Typ11_Fehleranalyse” relating mileage to suitable variables. Derive recommendations for OEM1 based on this model.

#### Initial Data exploration and preparation steps

Before building the regression model, we started by examining the structure of the dataset using the following methods:

1. `df_fail.head()`
2. `df_fail.columns.tolist()`
3. `df_fail.dtypes`

This revealed a problem: At first, the entire dataset appeared to be imported into a single column, with the first row lookin like:
>`,"X","X1","ID_Fahrzeug","Herstellernummer","Werksnummer", ...`

This indacated that:
- The data is comma-separated with all values enclosed in quotes
- Pandas default settings (`sep=","`) were not sufficient to parse it correctly

We used the `quotechar` parameter in `read_csv` to handle this. We also checked and converted the numerical columns (`fuel`, `days`, `Fehlerhaft_Fahrleistung`) with `pd.to_numeric(..., errors="coerce")` to avoid type errors in the model later. 

Only after validating and cleaning the dataset structure did we preceed with modelling.

#### Data preparation

To begin with, we import the dataset. The CSV file uses **comma-separated values with quoted entries**, so we use `sep=","` and `quotechar='"'` to ensure correct parsing.  
We select the following variables for modeling:

- **Dependent variable (`y`)**: `Fehlerhaft_Fahrleistung` (mileage at failure)
- **Predictors (`X`)**:
  - `days`: age of the vehicle in days
  - `fuel`: continuous metric (e.g. fuel consumption or energy index)
  - `engine`: engine type (categorical: small, medium, etc.)
  - `Werksnummer`: production plant (categorical)

We clean and transform the data as follows:
- Convert numerical columns explicitly to numeric types (with `errors="coerce"`).
- Convert categorical variables into dummy variables using `pd.get_dummies()`.
- Drop rows with missing values to avoid model errors.

In [ ]:
import pandas as pd
import statsmodels.api as sm

# Load CSV with proper settings
df_fail = pd.read_csv("data/Fahrzeug/Fahrzeuge_OEM1_Typ11_Fehleranalyse.csv", sep=",", quotechar='"')

# Optional: convert date column
df_fail["Fehlerhaft_Datum"] = pd.to_datetime(df_fail["Fehlerhaft_Datum"], errors="coerce")

# Define y (target) and X (predictors)
y = pd.to_numeric(df_fail["Fehlerhaft_Fahrleistung"], errors="coerce")
X = df_fail[["days", "fuel", "engine", "Werksnummer"]].copy()

# Ensure numeric types
for col in ["days", "fuel"]:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Set categorical columns
X["engine"] = X["engine"].astype("category")
X["Werksnummer"] = X["Werksnummer"].astype("category")

# Create dummy variables
X = pd.get_dummies(X, columns=["engine", "Werksnummer"], drop_first=True, dtype=float)

# Drop rows with NaNs in X or y
mask = X.notna().all(axis=1) & y.notna()
X_clean = X.loc[mask]
y_clean = y.loc[mask]

# Add constant for intercept
X_const = sm.add_constant(X_clean, has_constant="add")

### Code explanation
- `pd.get_dummies()` creates dummy variables for categorical predictors.
  - `drop_first=True` avoids the dummy variable trap (perfect multicollinearity).
- `add_constant()` adds an intercept to the model.
- Missing values (`NaN`) are removed to avoid runtime errors during model fitting.

In [ ]:
# --- Fit the model ---
model = sm.OLS(y_clean, X_const).fit()
model.summary()

### Results – Model Summary

The regression model was successfully fitted with **198,069 observations**. Key results:

- **R² = 0.484** → The model explains about 48.4% of the variance in vehicle mileage.
- **days (age)** → Not statistically significant (p = 0.501) → age alone doesn't explain mileage.
- **fuel** → Strong, positive, and highly significant coefficient.
- **engine_medium / engine_small** → Positive and significant → small engines accumulate significantly more mileage than large engines.
- **Werksnummer_12** → Not significant → no meaningful difference between plants.

#### Full coefficient table:

| Variable          | Coefficient   | p-value | Interpretation |
|-------------------|---------------|---------|----------------|
| days              | -0.0605       | 0.501   | No significant effect |
| fuel              | +5235         | < 0.001 | Higher fuel metric → more mileage |
| engine_medium     | +7848         | < 0.001 | More mileage vs. large engines |
| engine_small      | +11160        | < 0.001 | More mileage vs. large engines |
| Werksnummer_12    | +72           | 0.158   | No meaningful effect |

#### Model Notes and Limitations:

- The regression model outputs a Condition Number of ~17,700, indicating moderate multicollinearity or scaling issues. However, no implausible coefficients or signs of instability are present.
- Standard errors are computed under the assumption of homoskedasticity. For robustness, an alternative model with HC3 robust standard errors can be computed if needed.

#### Conclusion & Recommendations for OEM1

Based on the regression results, the following conclusions and strategic recommendations can be made:

- **Engine type is a strong predictor of mileage.**  
  Vehicles with **small engines** tend to accumulate the **most mileage**, followed by **medium engines**, with **large engines** used less intensively.  
  ⮕ OEM1 should adapt service intervals and warranty policies based on engine type.

- **Fuel metric (consumption/efficiency?) is highly correlated with mileage.**  
  ⮕ Further analysis could reveal whether fuel-efficient or high-usage vehicles drive this trend.

- **Vehicle age does not significantly explain mileage in this dataset.**  
  ⮕ Usage type and context (e.g. fleet vs. private) may be more relevant than age.

- **No plant-specific effects were observed.**  
  ⮕ This is a positive indicator of **uniform quality** across production locations.

**Overall**, this model helps OEM1 identify which vehicle types require more intensive usage tracking, servicing, and long-term support.

## Task 6 – Hit and Run Accident Investigation

### Task description:

On 11.08.2010, there was a hit-and-run accident. The license plate of the car involved is unknown. The police have asked for your assistance, as you work for the Federal Motor Transport Authority, to find out where the vehicle with body part number “K5-112-1122-79” was registered.

### Initial Considerations & Strategy:

As there is no direct link between a component and the registration data, the traceability must be performed in multiple logical steps by joining data across several files.

The iterative process was developed as follows:

1. Search for the component in the file `Bestandteile_Fahrzeuge_OEM1_Typ12.csv` to identify the corresponding `ID_Fahrzeug` (vehicle ID).
2. Using this vehicle ID, search in the file `Zulassungen_alle_Fahrzeuge.csv` to find the municipality and date of registration.

The following code cells document this procedure step-by-step.


In [ ]:
import pandas as pd

# Load relevant data files
df_components = pd.read_csv("data/Komponente/Komponente_K5.csv", sep=",", quotechar='"', encoding="utf-8", engine="python")
df_component_parts = pd.read_csv("data/Komponente/Bestandteile_Komponente_K5.csv", sep=";")
df_vehicle_parts = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv", sep=";")
df_registrations = pd.read_csv("data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv", sep=";")

In [ ]:
# Define the component ID to trace
target_component = "K5-112-1122-79"

# Step 1: Search for the vehicle that contains this component
vehicle_match = df_vehicle_parts[df_vehicle_parts["ID_Karosserie"] == target_component]

if not vehicle_match.empty:
    vehicle_id = vehicle_match["ID_Fahrzeug"].values[0]
    print(f"Identified vehicle ID: {vehicle_id}")
else:
    print("No matching vehicle found for the specified component.")

In [ ]:
# Step 2: Look up registration data for the identified vehicle
if not vehicle_match.empty:
    registration_match = df_registrations[df_registrations["IDNummer"] == vehicle_id]
    
    if not registration_match.empty:
        print("Registration details:")
        print(registration_match)
    else:
        print("No registration found for the identified vehicle.")

### Resuluts:

The component `K5-112-1122-79` was installed in the vehicle with ID `12-1-12-82`.  
This vehicle was registered on **2009-01-02** in the city of **ASCHERSLEBEN**.